# 03 · Features and pipeline: each addition judged on June
**Owner: P2** · Status: `[x]`

Rules: stateless features live in `src/features.py`; anything learned lives inside the `Pipeline` (`src/models.py`) and is fitted on **May only**. June scores every choice. **July is not loaded here.** Report the features that lost too.

In [ ]:
import sys; sys.path.insert(0, "..")          # repo root, so `import src...` works
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from src.config import TARGET
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 140)

import itertools, time
from src.clean import build_datasets
from src.features import FEATURE_SETS
from src.models import make_pipeline, make_model
from src.evaluate import metrics, bootstrap_vs_baseline, save_results

D = build_datasets()
train, val = D["train"], D["val"]            # NB03 never touches D["test"]
del D
ytr, yva = train[TARGET], val[TARGET]
days_va = val.tpep_pickup_datetime.dt.date   # bootstrap blocks
print(len(train), len(val))

def june(model_name, fs, **params):
    pipe = make_pipeline(make_model(model_name, **params), fs).fit(train, ytr)
    return pipe, pipe.predict(val)

## 1. Ablation: feature sets × models (train May, score June)
New sets beyond the skeleton (f0–f4):
- **f5_zones**: pickup and drop-off zone IDs, **target-encoded inside the pipeline** (sklearn `TargetEncoder`, 5-fold cross-fitted on May; June gets May's encodings).
- **f6_pair**: plus the zone *pair*, target-encoded the same way (the zone-pair mean is never computed on June/July).
- **f7_time_x**: plus hour × weekday/weekend, and distance × (time band × weekday/weekend) so the per-mile price can vary with traffic.

Three model families: OLS, a single decision tree (defaults, tuned in §2), and histogram gradient boosting (sklearn, defaults, tuned in §3).

In [ ]:
rows = []
for fs in FEATURE_SETS:
    for name in ["linear", "tree", "hgb"]:
        pipe, p = june(name, fs)
        rows.append(dict(feature_set=fs, model=name, n_features=pipe[:-1].transform(val.head(5)).shape[1],
                         **{k: round(v, 3) for k, v in metrics(yva, p).items() if k != "n"}))
ablation = pd.DataFrame(rows)
ablation.to_csv("../results/june_ablation.csv", index=False)
ablation.pivot(index="feature_set", columns="model", values=["mae", "rmse"])

FILL_ABLATION

## 2. Tune the tree on June
The skeleton grid's best (depth 12, leaf 50) sat on the edge of the grid, so the grid is extended. Feature set: f7_time_x (best for the tree in §1).

In [ ]:
res = []
for depth, leaf in itertools.product([8, 10, 12, 14, 16, 20], [10, 20, 50, 100, 200]):
    _, p = june("tree", "f7_time_x", max_depth=depth, min_samples_leaf=leaf)
    res.append(dict(depth=depth, leaf=leaf, **{k: round(v, 3) for k, v in metrics(yva, p).items() if k != "n"}))
tree_grid = pd.DataFrame(res).sort_values("mae")
tree_grid.to_csv("../results/june_tree_grid.csv", index=False)
tree_grid.head(8)

FILL_TREE

## 3. Tune gradient boosting on June
Stage 1: loss × learning rate × leaves × iterations × min leaf size (48 fits). `absolute_error` targets the conditional median, which is what MAE rewards; `squared_error` targets the mean. Stage 2 extends past any edge optimum from stage 1.

**Rule, fixed before the grid ran:** because many neighbouring configurations differ by cents, we take the *cheapest* configuration (fewest leaves, then fewest iterations) within \$0.01 of the best June MAE, rather than the noisy argmin.

In [ ]:
t0 = time.time(); res = []
for loss, lr, leaves, iters, msl in itertools.product(["squared_error", "absolute_error"], [0.05, 0.1], [31, 63, 127], [300, 600], [20, 100]):
    _, p = june("hgb", "f7_time_x", loss=loss, learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl)
    res.append(dict(loss=loss, learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl,
                    **{k: round(v, 4) for k, v in metrics(yva, p).items() if k != "n"}))
hgb1 = pd.DataFrame(res).sort_values("mae")
print(f"stage 1: {time.time() - t0:.0f}s"); display(hgb1.head(8)); display(hgb1.groupby("loss")["mae"].agg(["min", "median"]))

In [ ]:
t0 = time.time(); res = []
for lr, leaves, iters, msl in itertools.product([0.05, 0.1], [127, 255], [600, 1000], [100, 300]):
    _, p = june("hgb", "f7_time_x", loss="absolute_error", learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl)
    res.append(dict(loss="absolute_error", learning_rate=lr, max_leaf_nodes=leaves, max_iter=iters, min_samples_leaf=msl,
                    **{k: round(v, 4) for k, v in metrics(yva, p).items() if k != "n"}))
hgb = pd.concat([hgb1, pd.DataFrame(res)]).drop_duplicates(subset=["loss", "learning_rate", "max_leaf_nodes", "max_iter", "min_samples_leaf"]).sort_values("mae")
hgb.to_csv("../results/june_hgb_grid.csv", index=False)
print(f"stage 2: {time.time() - t0:.0f}s, best MAE {hgb.mae.min():.4f}, range of the top 10: {hgb.mae.head(10).min():.4f}-{hgb.mae.head(10).max():.4f}")
near = hgb[hgb.mae <= hgb.mae.min() + 0.01].sort_values(["max_leaf_nodes", "max_iter", "mae"])
display(near)
chosen = near.iloc[0][["loss", "learning_rate", "max_leaf_nodes", "max_iter", "min_samples_leaf"]].to_dict()
chosen = {k: (v.item() if hasattr(v, "item") else v) for k, v in chosen.items()}
print("chosen:", chosen)

FILL_HGB

## 4. Model selection on June, with uncertainty
**Rule, fixed before this section ran:** take the lowest June MAE; if its gain over the best *simpler* candidate has a 95% day-block bootstrap CI that includes 0, take the simpler one. Candidates (each at its best feature set and tuned params): OLS, tuned tree, tuned HGB, plus the rate rule baselines.

The bootstrap resamples whole June days (30 blocks), because trips on the same day share weather, traffic and events. **Caveat:** June has now been used for feature choice, tuning *and* this comparison, so the winner's June MAE is optimistic. July (NB04) is the honest estimate.

In [ ]:
from src.models import RateRule, TLC_START, TLC_PER_MILE
best_tree = tree_grid.iloc[0]
cands = {
    "rate rule (TLC $3 + $3.50/mi)": RateRule(TLC_START, TLC_PER_MILE).fit(train, ytr).predict(val),
    "rate rule (fit on May)": RateRule().fit(train, ytr).predict(val),
    "linear f7": june("linear", "f7_time_x")[1],
    f"tree f7 (depth {best_tree.depth}, leaf {best_tree.leaf})": june("tree", "f7_time_x", max_depth=int(best_tree.depth), min_samples_leaf=int(best_tree.leaf))[1],
    "hgb f7 (chosen params)": june("hgb", "f7_time_x", **chosen)[1],
}
sel = pd.DataFrame({k: metrics(yva, p) for k, p in cands.items()}).T.drop(columns="n").round(3)
display(sel)
names = list(cands)
best, simpler = names[-1], names[-2]
cmp = {f"{best} vs {o}": bootstrap_vs_baseline(yva, cands[best], cands[o], groups=days_va, n_boot=2000).loc[["mae_diff", "rmse_diff"]]
       for o in [simpler, "linear f7", "rate rule (fit on May)"]}
pd.concat(cmp).round(3)

FILL_SELECT

## 5. Features that lost / did nothing
From the ablation table (§1), June MAE change when the set is added, per model:
FILL_LOST

## 6. Freeze (Gate 2)
FILL_FREEZE

In [ ]:
from src.config import FINAL_FEATURE_SET, FINAL_MODEL, FINAL_PARAMS
assert (FINAL_MODEL, FINAL_FEATURE_SET) == ("hgb", "f7_time_x") and FINAL_PARAMS == chosen, "config.py does not match the June selection"
save_results({"feature_set": FINAL_FEATURE_SET, "model": FINAL_MODEL, "params": FINAL_PARAMS,
              "june_selection_table": sel.to_dict(orient="index"),
              "june_bootstrap": {k: v.to_dict(orient="index") for k, v in cmp.items()}}, "gate2_freeze.json")
print("frozen:", FINAL_FEATURE_SET, FINAL_MODEL, FINAL_PARAMS)